In [31]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Intro to Managed Agents API on Agent Platform (Python)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/managed-agents/intro_managed_agents_python.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%agents%2Fmanaged-agents%2Fintro_managed_agents_python.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/agents/managed-agents/intro_managed_agents_python.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/managed-agents/intro_managed_agents_python.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/managed-agents/intro_managed_agents_python.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/managed-agents/intro_managed_agents_python.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/managed-agents/intro_managed_agents_python.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/managed-agents/intro_managed_agents_python.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/managed-agents/intro_managed_agents_python.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>


| Authors |
| --- |
| [Eric Schmidt](https://github.com/cloude-google) |

### Overview

This notebook demonstrates operations for **Managed Agents API on Gemini Enterprise Agent Platform** using the Gen AI SDK for Python.

Covering:
- **Managed Agents API**: Create, list, get, and delete custom agents
- **Interactions API**: Interact with Antigravity (1P) and custom agents
- **Environment Features**: Session state management, MCP tools, skills


**Note:** The Managed Agents API is in **Preview**.
* Features and schemas are subject to change.
* They are not intended for production applications.
* It is highly recommended to run this sample in an isolated development or testing project.

For complete Manged Agents API documentation please visit: https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/managed-agents.

## Getting Started

### Install Gen AI SDK for Python

In [32]:
%pip install --upgrade --quiet "google-genai>=2.0.0"

⚠️ Note: Ignore pip dependency errors.

### Import Libraries

In [33]:
import os
import sys
import requests

from google import genai

### Authenticate your Notebook Environment

If you are running this notebook in **Google Colab**, execute the cell below to authenticate.

In [34]:
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud Project Information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [35]:
# fmt: off
PROJECT_ID = "project-7a7c7558-fc78-4a7f-8ae"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# TODO: Replace "" with your actual Google Cloud Project ID (e.g., "my-gcp-project-123").
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.getenv("GOOGLE_CLOUD_PROJECT"))

LOCATION = "global"

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

## Project Validation

Before provisioning agents or starting conversations, verify that your Google Cloud project and permissions meet the platform requirements.

The following diagnostic helper validates:
*   **Authentication**: Confirms active Application Default Credentials (ADC).
*   **API Enablement**: Verifies **`aiplatform.googleapis.com`** is enabled.
*   **Service Account Role**: Confirms the Google-managed AI Platform service agent has the **`roles/aiplatform.serviceAgent`** role (required for container sandbox and GCS bucket orchestration).
*   **User Access**: Confirms your identity is authorized with **`roles/aiplatform.user`**, **`roles/aiplatform.admin`**, or **`roles/owner`**.

> [!NOTE]
> If any check fails, the diagnostic helper will print specific commands to resolve the issue.

In [36]:
def check_project_settings(project_id: str) -> None:
    """Validates GCP project settings, APIs, and IAM roles for AI Platform."""

    # Extract Token
    printed_token = !gcloud auth application-default print-access-token
    token = printed_token[0] if printed_token else None

    if token:
        print("✅ Success on token creation.")
    else:
        print("❌ Token creation failed.")
        return

    # Extract project number
    project_info_response = !gcloud projects describe {project_id} --format="value(projectNumber)"
    if not project_info_response:
        print("❌ Failed to retrieve project number.")
        return

    project_number = project_info_response[0]
    print(f"✅ The project number for {project_id} is: {project_number}")

    # Is aiplatform enabled
    api_name = "aiplatform.googleapis.com"
    result = !gcloud services list --project={project_id} --enabled --filter="name:{api_name}" --format="value(name)"

    if any(api_name in service for service in result):
        print(f"✅ {api_name} is enabled.")
    else:
        print(f"❌ {api_name} is NOT enabled. You may need to run:")
        print(f"   !gcloud services enable {api_name} --project={project_id}")

    # Does service account have needed bindings
    sa_email = f"service-{project_number}@gcp-sa-aiplatform.iam.gserviceaccount.com"
    sa_info_response = !gcloud projects get-iam-policy {project_id} \
        --flatten="bindings[].members" \
        --filter="bindings.members:serviceAccount:{sa_email}" \
        --format="value(bindings.role)"

    if any("aiplatform.serviceAgent" in role for role in sa_info_response):
        print("✅ The service account has the aiplatform.serviceAgent role.")
    else:
        print("❌ The service account does NOT have the aiplatform.serviceAgent role.")
        print("Attempting to add role...")
        attempt_add_role = !gcloud projects add-iam-policy-binding {project_id} \
                            --member="serviceAccount:{sa_email}" \
                            --role="roles/aiplatform.serviceAgent"

    # Extract email from token
    email = None
    try:
        token_info_response = requests.get(f"https://oauth2.googleapis.com/tokeninfo?access_token={token}", timeout=10)

        if token_info_response.status_code == 200:
            data = token_info_response.json()
            # The email will only be present if the "userinfo.email" scope was requested
            email = data.get("email")
            print(f"Token belongs to: {email}")
        else:
            print("Invalid or expired token.")
            return
    except requests.RequestException as e:
        print(f"Failed to reach tokeninfo API: {e}")
        return

    if not email:
        print("Email scope not requested; cannot verify user IAM roles.")
        return

    # Does user have needed bindings
    user_roles_response = !gcloud projects get-iam-policy {project_id} \
        --flatten="bindings[].members" \
        --filter="bindings.members:user:{email}" \
        --format="value(bindings.role)"

    print(f"Raw roles found: {user_roles_response}")

    if not user_roles_response:
        print(f"❌ The user {email} has no direct project-level roles assigned.")
        return

    # Check for owner, admin, and user roles
    is_owner = any("roles/owner" in role for role in user_roles_response)
    is_ai_admin = any("aiplatform.admin" in role for role in user_roles_response)
    is_ai_user = any("aiplatform.user" in role for role in user_roles_response)

    if is_owner:
        print(f"✅ The user {email} is a project Owner, which inherits all AI Platform permissions.")
    elif is_ai_admin:
        print(f"✅ The user {email} has the explicit aiplatform.admin role.")
    elif is_ai_user:
        print(f"✅ The user {email} has the explicit aiplatform.user role (standard user access).")
    else:
        print(f"❌ The user {email} does NOT have the aiplatform.admin, aiplatform.user, or owner role.")

check_project_settings(PROJECT_ID)

✅ Success on token creation.
✅ The project number for project-7a7c7558-fc78-4a7f-8ae is: 554092039142
✅ aiplatform.googleapis.com is enabled.
✅ The service account has the aiplatform.serviceAgent role.
Token belongs to: aleknodler@gmail.com
Raw roles found: ['roles/owner']
✅ The user aleknodler@gmail.com is a project Owner, which inherits all AI Platform permissions.


### Create a Google Cloud Storage Bucket

To demonstrate mounting remote workspace directories into an agent container, you need a target **Google Cloud Storage (GCS)** bucket.

Create new bucket that you will use for testing.

The following cell programmatically provisions a new GCS bucket using the `gcloud` CLI. This bucket will be mounted as a local workspace path when creating your custom agent in the next step.

In [37]:
AGENT_GCS_BUCKET = "project-7a7c7558-fc78-4a7f-8ae"  # @param {type:"string"}
create_response = !gcloud storage buckets create gs://{AGENT_GCS_BUCKET} --project={PROJECT_ID}
print(create_response)

['Creating gs://project-7a7c7558-fc78-4a7f-8ae/...', '\x1bERROR:\x1b (gcloud.storage.buckets.create) HTTPError 409: Your previous request to create the named bucket succeeded and you already own it.']


## Managed Agents API — Create, List, Get, Delete

The **Managed Agents API** serves as the **Control Plane** of the platform. It allows you to provision, configure, retrieve, and manage stateful, reusable agent resources that persist securely within your Google Cloud project.

Each custom agent is defined by extending a `base_agent` (such as `antigravity-preview-05-2026`) and configuring:
*   **System Instructions**: Tailor the agent's expertise and behavior policies.
*   **Built-in Tools**: Enable local capabilities like code execution, filesystem operations, or Google Search.
*   **Workspace Mounts**: Attach Google Cloud Storage (GCS) directories as sandboxed local paths.
*   **Third-Party Integrations**: Connect secure Model Context Protocol (MCP) servers.
*   **Skill Registries**: Mount reusable domain-expert instructions from the platform's Skill Registry.


### 1. Create a Custom Agent with GCS Mount and Default Tools

To start, create a custom agent resource. This agent will extend the foundational base agent with a targeted system instruction, enable built-in developer tools, and mount your Google Cloud Storage (GCS) bucket as a local directory path in its sandboxed environment.

In [38]:
response = client.interactions.create(
    agent=AGENT_ID,
    input="Olá! Você é o meu assistente de programação? Quais ferramentas você tem ativas?",
    background=True
)

print(response)


status='in_progress' model=None agent='sdk-demo-agent-d12dc68d' id='ChAxZTA1N2QzNmQ1ZTBiNDUyEAgaATAqBG1haW4' created=None updated=None system_instruction=None tools=None usage=None response_modalities=None response_mime_type=None previous_interaction_id=None environment_id=None service_tier=None webhook_config=None steps=None response_format=None environment=None generation_config=None agent_config=None safety_settings=None labels=None input=None output_text='' output_image=None output_audio=None output_video=None object='interaction'


In [39]:
resultado = client.interactions.get(id=response.id)

print("--- RESPOSTA DO AGENTE ---")
if hasattr(resultado, 'output') and resultado.output:
    print(resultado.output)
else:
    print(resultado)


--- RESPOSTA DO AGENTE ---
status='in_progress' model=None agent='sdk-demo-agent-d12dc68d' id='ChAxZTA1N2QzNmQ1ZTBiNDUyEAgaATAqBG1haW4' created=None updated=None system_instruction=None tools=None usage=None response_modalities=None response_mime_type=None previous_interaction_id=None environment_id='env_CAEQgICAgIDwiapTGiBmYWMzYjI0M2QxN2E0NjJlOWRiYmJhZDM3OTM2ZDQzZA' service_tier=None webhook_config=None steps=[UserInputStep(content=[TextContent(text='Olá! Você é o meu assistente de programação? Quais ferramentas você tem ativas?', annotations=None, type='text')], type='user_input')] response_format=None environment=None generation_config=None agent_config=None safety_settings=None labels=None input=None output_text='' output_image=None output_audio=None output_video=None object='interaction'


In [40]:
import json

try:
    dados = json.loads(resultado.model_dump_json())
    chunks = dados.get("response_chunks", [])
    for chunk in chunks:
        texto = chunk.get("interaction_chunk", {}).get("response", {}).get("text", "")
        if texto:
            print(texto)
except Exception:
    print(resultado)


In [41]:
import json

dados = json.loads(resultado.model_dump_json())
print(json.dumps(dados, indent=2, ensure_ascii=False))


{
  "status": "in_progress",
  "agent": "sdk-demo-agent-d12dc68d",
  "id": "ChAxZTA1N2QzNmQ1ZTBiNDUyEAgaATAqBG1haW4",
  "environment_id": "env_CAEQgICAgIDwiapTGiBmYWMzYjI0M2QxN2E0NjJlOWRiYmJhZDM3OTM2ZDQzZA",
  "steps": [
    {
      "content": [
        {
          "text": "Olá! Você é o meu assistente de programação? Quais ferramentas você tem ativas?",
          "type": "text"
        }
      ],
      "type": "user_input"
    }
  ],
  "output_text": "",
  "object": "interaction"
}


In [42]:
import uuid

AGENT_ID = f"sdk-demo-agent-{uuid.uuid4().hex[:8]}"

agent = client.agents.create(
    id=AGENT_ID,
    base_agent="antigravity-preview-05-2026",
    description="A demo agent created with the Python SDK.",
    system_instruction="You are a helpful coding assistant. Write clean, well-documented Python code.",
    tools=[
        {"type": "code_execution"},
        {"type": "google_search"},
        {"type": "url_context"},
    ],
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "gcs",
                "source": "gs://"+AGENT_GCS_BUCKET,
                "target": "/.agent",
            }
        ],
        "network": {
            "allowlist": [{"domain": "*"}]
        },
    },
)

# Note: Agent creation is an asynchronous
# You can poll creation status by calling client.agents.get()

print(f"Agent created: {AGENT_ID}")
print(agent)

Agent created: sdk-demo-agent-49b38308
base_agent=None base_environment=None description=None id=None system_instruction=None tools=None name='projects/554092039142/locations/global/agents/sdk-demo-agent-49b38308/operations/1062214544094396416' metadata={'@type': 'type.googleapis.com/google.cloud.aiplatform.v1beta1.CreateAgentOperationMetadata', 'genericMetadata': {'createTime': '2026-09-29T22:16:57.878561Z', 'updateTime': '2026-09-29T22:16:57.878561Z'}}


In [53]:
import time
import traceback
from google.api_core import exceptions # Import exceptions for NotFoundError

print(f"Agent creation operation name: {agent.name}") # This still prints the LRO name

try:
    # We cannot call agent.result() directly as 'Agent' object has no attribute 'result'.
    # We will poll for the agent's existence using client.agents.get(id=AGENT_ID).

    polling_interval = 5 # seconds
    max_retries = 12 # 1 minute timeout (12 * 5 seconds)
    retries = 0
    completed_agent = None

    print(f"Polling for agent '{AGENT_ID}' to be created...")
    while retries < max_retries:
        try:
            completed_agent = client.agents.get(id=AGENT_ID)
            # If get() succeeds, the agent is created.
            break
        except exceptions.NotFound:
            print(f"Agent '{AGENT_ID}' not yet found. Retrying in {polling_interval} seconds...")
            time.sleep(polling_interval)
            retries += 1
        except Exception as e:
            print(f"An unexpected error occurred while polling: {type(e).__name__} - {e}")
            traceback.print_exc()
            raise # Re-raise unexpected errors

    if completed_agent and hasattr(completed_agent, 'id'):
        print(f"\nAgent created successfully: {completed_agent.id}")
        print(completed_agent)
    elif completed_agent:
        print("\nAgent creation completed, but the resulting agent object was invalid or missing ID.")
        print(f"Resulting object: {completed_agent}")
    else:
        print(f"\nAgent creation timed out after {max_retries * polling_interval} seconds. Agent '{AGENT_ID}' not found.")


except Exception as e:
    print(f"Error getting agent creation result: {type(e).__name__} - {e}")
    traceback.print_exc()
    # The inner polling loop should handle most errors, this outer block is for
    # anything that escapes the polling.

Agent creation operation name: projects/554092039142/locations/global/agents/sdk-demo-agent-49b38308/operations/1062214544094396416
Polling for agent 'sdk-demo-agent-49b38308' to be created...

Agent created successfully: sdk-demo-agent-49b38308
base_agent='antigravity-preview-05-2026' base_environment=Environment(environment_id=None, network=Allowlist(allowlist=[AllowlistEntry(domain='*', transform=None)]), sources=[Source(content=None, encoding=None, source='gs://project-7a7c7558-fc78-4a7f-8ae', target='/.agent', type='gcs')], type='remote') description='A demo agent created with the Python SDK.' id='sdk-demo-agent-49b38308' system_instruction='You are a helpful coding assistant. Write clean, well-documented Python code.' tools=[CodeExecution(type='code_execution'), GoogleSearch(search_types=None, type='google_search'), URLContext(type='url_context')] name='projects/project-7a7c7558-fc78-4a7f-8ae/locations/global/agents/sdk-demo-agent-49b38308' created='2026-09-29T22:16:57.878561Z' u

### Teste o agente recém-criado

In [54]:
stream = client.interactions.create(
    agent=AGENT_ID,
    input="Olá! Você pode me dizer qual é o seu propósito?",
    stream=True,
    background=True,
    store=True,
)

print(f"Resposta do Agente Personalizado ({AGENT_ID}):")
print("=" * 60)

for event in stream:
    print(event)

Resposta do Agente Personalizado (sdk-demo-agent-49b38308):
interaction=InteractionSseEventInteraction(id='ChA0YWQzMzQ0YjRjZmI3NmRlEAgaATAqBG1haW4', status='in_progress', object='interaction', model=None, agent='sdk-demo-agent-49b38308', created=None, updated=None, service_tier=None, usage=None, steps=None) event_type='interaction.created' event_id=None metadata=None
interaction_id='ChA0YWQzMzQ0YjRjZmI3NmRlEAgaATAqBG1haW4' status='in_progress' event_id=None event_type='interaction.status_update' metadata=None
index=0 step=ModelOutputStep(content=None, error=None, type='model_output') event_id=None event_type='step.start' metadata=None
delta=TextDelta(text='Olá! Eu sou o Antigravity, um assistente de inteligência artificial avançado', type='text') index=0 event_id='v1_MF90ZXh0XzE3OTA3MjE5MTI5NDYzOTk=' event_type='step.delta' metadata=None
delta=TextDelta(text=' desenvolvido pelo Google. \n\nO meu propósito é colaborar com você em tarefas avançadas de programação, como:\n- Criar novos pr

#### **Option A**: Create an Agent with Model Context Protocol (MCP) Tools

You can extend an agent's toolsuite beyond standard built-in tools by linking secure, remote **Model Context Protocol (MCP)** servers. Under the `tools` payload, supply a target dictionary containing the URL of your MCP server and any mandatory security headers required for authentication.

In [55]:
import time

print(f"SDK version: {genai.__version__}")
print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")

MCP_AGENT_ID = f"mcp-agent-{uuid.uuid4().hex[:8]}"

mcp_agent = client.agents.create(
    id=MCP_AGENT_ID,
    base_agent="antigravity-preview-05-2026",
    description="An agent with MCP tool access for code search.",
    system_instruction="You are a helpful assistant with access to code search tools.",
    tools=[
        {
            "type": "mcp_server",
            "name": "grep-search",
            "url": "https://mcp.grep.app",
        }
    ],
)

print(f"MCP Agent created: {MCP_AGENT_ID}")
print(mcp_agent)

time.sleep(10)
response = client.agents.list()

if response.agents:
    print(f"Found {len(response.agents)} agent(s):\n")
    for i, a in enumerate(response.agents, 1):
        print(f"  [{i}] {a.id}")
        print(f"      Base Agent:   {a.base_agent or '—'}")
        print(f"      Description:  {a.description or '—'}")
        tools_str = ", ".join(t.type for t in a.tools) if a.tools else "None"
        print(f"      Tools:        {tools_str}")
        print()
    if response.next_page_token:
        print(f"More results available (next_page_token: {response.next_page_token})")
else:
    print("No agents found.")


SDK version: 2.12.1
Project: project-7a7c7558-fc78-4a7f-8ae
Location: global
MCP Agent created: mcp-agent-8165a8f3
base_agent=None base_environment=None description=None id=None system_instruction=None tools=None name='projects/554092039142/locations/global/agents/mcp-agent-8165a8f3/operations/622550630472351744' metadata={'@type': 'type.googleapis.com/google.cloud.aiplatform.v1beta1.CreateAgentOperationMetadata', 'genericMetadata': {'createTime': '2026-09-29T22:45:23.437762Z', 'updateTime': '2026-09-29T22:45:23.437762Z'}}
Found 4 agent(s):

  [1] mcp-agent-8165a8f3
      Base Agent:   antigravity-preview-05-2026
      Description:  An agent with MCP tool access for code search.
      Tools:        mcp_server

  [2] sdk-demo-agent-49b38308
      Base Agent:   antigravity-preview-05-2026
      Description:  A demo agent created with the Python SDK.
      Tools:        code_execution, google_search, url_context

  [3] sdk-demo-agent-d12dc68d
      Base Agent:   antigravity-preview-05-202

#### **Option B**: Mount Skills from the Skill Registry

To build highly specialized domain-expert agents, you can mount structured instruction packages directly from your central **Skill Registry**. By referencing the unique registered skill name, the agent will dynamically discover and parse its instructions.

> **NOTE**: Replace `SKILL_RESOURCE_NAME` with an actual skill path from your project.
> Example: `projects/your-project/locations/us-central1/skills/your-skill`


In [57]:
SKILL_AGENT_ID = f"skill-agent-{uuid.uuid4().hex[:8]}"
SKILL_RESOURCE_NAME = "projects/your-project/locations/us-central1/skills/your-skill"  # @param {type:"string"}

skill_agent = client.agents.create(
    id=SKILL_AGENT_ID,
    base_agent="antigravity-preview-05-2026",
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "skill_registry",
                "source": SKILL_RESOURCE_NAME,
                "target": "./skills",
            }
        ],
        "network": {
            "allowlist": [{"domain": "*"}]
        },
    },
)

print(f"Skill agent created: {SKILL_AGENT_ID}")
print(skill_agent)


BadRequestError: Error code: 400 - {'error': {'code': 400, 'message': 'Skill projects/your-project/locations/us-central1/skills/your-skill does not belong to the serving project.', 'status': 'INVALID_ARGUMENT'}}

#### Create a GCS Bucket to Stage Skill Packages

Create a dedicated Google Cloud Storage bucket to host and serve raw skill packages for your custom agents.

In [58]:
AGENT_GCS_BUCKET = ""  # @param {type:"string"}
create_response = !gcloud storage buckets create gs://{AGENT_GCS_BUCKET} --project={PROJECT_ID}
print(create_response)

['\x1bERROR:\x1b (gcloud.storage.buckets.create) "gcloud storage buckets create" only accepts bucket URLs.', 'Example: "gs://bucket"', 'Received: "gs://"']


#### **Option C**: Mount Skill Packages from a GCS Path

Alternatively, you can configure an agent to dynamically load raw skill packages stored directly inside a Google Cloud Storage (GCS) directory. This lets you stage and test new skills directly from cloud storage without having to register them in the central Skill Registry first.

In [59]:
AGENT_SKILL_GCS_BUCKET = ""  # @param {type:"string"}
create_response = !gcloud storage buckets create gs://{AGENT_SKILL_GCS_BUCKET} --project={PROJECT_ID}
print(create_response)

GCS_SKILL_AGENT_ID = f"gcs-skill-agent-{uuid.uuid4().hex[:8]}"

gcs_skill_agent = client.agents.create(
    id=GCS_SKILL_AGENT_ID,
    base_agent="antigravity-preview-05-2026",
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "gcs",
                "source": "gs://"+AGENT_SKILL_GCS_BUCKET,
                "target": "./skills",
            }
        ],
        "network": {
            "allowlist": [{"domain": "*"}]
        },
    },
)

print(f"GCS skill agent created: {GCS_SKILL_AGENT_ID}")
print(gcs_skill_agent)


['\x1bERROR:\x1b (gcloud.storage.buckets.create) "gcloud storage buckets create" only accepts bucket URLs.', 'Example: "gs://bucket"', 'Received: "gs://"']


BadRequestError: Error code: 400 - {'error': {'code': 400, 'message': 'Invalid GCS source: `gs://`.', 'status': 'INVALID_ARGUMENT'}}

### 3. List Registered Agents

Retrieve a list of all custom agents provisioned and configured under your target Google Cloud project.

In [60]:
response = client.agents.list()

if response.agents:
    print(f"Found {len(response.agents)} agent(s):\n")
    for i, a in enumerate(response.agents, 1):
        print(f"  [{i}] {a.id}")
        print(f"      Base Agent:   {a.base_agent or '—'}")
        print(f"      Description:  {a.description or '—'}")
        tools_str = ", ".join(t.type for t in a.tools) if a.tools else "None"
        print(f"      Tools:        {tools_str}")
        print()
    if response.next_page_token:
        print(f"More results available (next_page_token: {response.next_page_token})")
else:
    print("No agents found.")


Found 4 agent(s):

  [1] mcp-agent-8165a8f3
      Base Agent:   antigravity-preview-05-2026
      Description:  An agent with MCP tool access for code search.
      Tools:        mcp_server

  [2] sdk-demo-agent-49b38308
      Base Agent:   antigravity-preview-05-2026
      Description:  A demo agent created with the Python SDK.
      Tools:        code_execution, google_search, url_context

  [3] sdk-demo-agent-d12dc68d
      Base Agent:   antigravity-preview-05-2026
      Description:  A demo agent created with the Python SDK.
      Tools:        code_execution, google_search, url_context

  [4] mcp-agent-3862a14b
      Base Agent:   antigravity-preview-05-2026
      Description:  An agent with MCP tool access for code search.
      Tools:        mcp_server



### 4. Retrieve a Specific Agent Config

Query the control plane using a specific `agent_id` to inspect its display name, active system instructions, mounted environments, or enabled tool configurations.

In [61]:
agent_details = client.agents.get(id=AGENT_ID)

print(f"Agent ID:          {agent_details.id}")
print(f"Base Agent:        {agent_details.base_agent}")
print(f"Description:       {agent_details.description}")
print(f"System Instruction: {agent_details.system_instruction[:100] if agent_details.system_instruction else '—'}...")
print(f"Tools:             {[t.type for t in agent_details.tools] if agent_details.tools else 'None'}")
print(f"Environment:       {agent_details.base_environment}")

Agent ID:          sdk-demo-agent-49b38308
Base Agent:        antigravity-preview-05-2026
Description:       A demo agent created with the Python SDK.
System Instruction: You are a helpful coding assistant. Write clean, well-documented Python code....
Tools:             ['code_execution', 'google_search', 'url_context']
Environment:       environment_id=None network=Allowlist(allowlist=[AllowlistEntry(domain='*', transform=None)]) sources=[Source(content=None, encoding=None, source='gs://project-7a7c7558-fc78-4a7f-8ae', target='/.agent', type='gcs')] type='remote'


### Update agent (REST only)

> **Note:** The `agents.update()` method is not yet available in the Python SDK. To update an agent's configuration (system instruction, tools, environment, skills), use the REST API with a `PATCH` request and `update_mask`.
>
> See the [Update an agent](https://cloud.google.com/gemini-enterprise-agent-platform/build/managed-agents/create-manage#update-an-agent) documentation for REST examples.


---
## Interactions API — Interact with Agents

The **Interactions API** is the data plane for communicating with agents at runtime. It supports streaming responses, environment management, and dynamic tool overrides.


### 1. Interact with the Foundational Base Agent

You can stream conversational interactions directly against the platform's pre-configured foundational `base_agent` (e.g., `antigravity-preview-05-2026`) without establishing a custom agent profile first.

In [62]:
stream = client.interactions.create(
    agent="antigravity-preview-05-2026",
    input="Who are you? Can you execute Python code? Show me an example.",
    environment={"type": "remote"},
    stream=True,
    background=True,
    store=True
)

print("Antigravity Agent Response (streaming):")
print("=" * 60)

for event in stream:
    print(event)


KeyboardInterrupt: 

### 2. Interact with your Custom Agent

Execute stateful, multi-turn conversations against the custom agent you provisioned on the control plane. This request automatically invokes all mounted capabilities, system instructions, and customized tool suites.

In [ ]:
stream = client.interactions.create(
    agent=AGENT_ID,
    input="Tell me the name of python packages used for data analysis.",
    stream=True,
    background=True,
    store=True,
)

print(f"Custom Agent ({AGENT_ID}) Response:")
print("=" * 60)

for event in stream:
    print(event)


---
## Session State with Environment IDs

By default, interactions are stateless. To maintain session state (files, installed packages, execution context) across multiple turns, reuse the **environment ID** (`env_id`) returned from the initial interaction.

The sandbox has a **7-day TTL** that resets with each new interaction.


#### Initiate the Session Environment

To maintain conversational state and persist runtime context (like environment variables and local filesystem files), initiate the first interaction with the custom agent. The platform will allocate a dedicated compute sandbox and return a unique `environment_id`.

In [ ]:
stream = client.interactions.create(
    agent="antigravity-preview-05-2026",
    input="Create a file called hello.txt with the content 'Hello from the sandbox!'",
    environment={"type": "remote"},
    stream=True,
    background=True,
    store=True,
)

env_id = None
for event in stream:
    print(event)
    # Extract environment_id from the completed event
    if hasattr(event, 'interaction') and hasattr(event.interaction, 'environment_id'):
        env_id = event.interaction.environment_id

print(f"\nEnvironment ID: {env_id}")


#### Continue the Thread (Reusing the Environment)

Execute subsequent interaction requests by passing the active `environment_id`. The platform will load the pre-allocated sandbox session, preserving variables, local filesystem modifications, and conversational context.

In [66]:
if env_id:
    stream = client.interactions.create(
        agent="antigravity-preview-05-2026",
        input="Read the file hello.txt and print its contents.",
        environment=env_id,
        stream=True,
        background=True,
        store=True,
    )

    print("Follow-up Response (same environment):")
    print("=" * 60)

    for event in stream:
        print(event)
else:
    print("No environment ID available. Run the previous cell first.")


NameError: name 'env_id' is not defined

---
## Override MCP Configurations at Interaction Time

You can dynamically override or add MCP server tools during an interaction without modifying the underlying agent configuration. This is useful for per-request tool customization.


### Dynamically Inject and Override MCP Servers

At execution time, you can dynamically override the agent's control-plane MCP configuration or inject entirely new temporary tools by supplying a modified server dictionary within the interaction request.

In [63]:
stream = client.interactions.create(
    agent=AGENT_ID,
    input="Use the grep tool to search for 'fibonacci' in github.",
    tools=[
        {
            "type": "mcp_server",
            "url": "https://mcp.grep.app",
            "name": "grep-search",
        }
    ],
    stream=True,
    background=True,
    store=True,
)

print("MCP Override Response:")
print("=" * 60)

for event in stream:
    print(event)


MCP Override Response:
interaction=InteractionSseEventInteraction(id='ChBlNzBmN2NmODJjMTZhNWRjEAgaATAqBG1haW4', status='in_progress', object='interaction', model=None, agent='sdk-demo-agent-49b38308', created=None, updated=None, service_tier=None, usage=None, steps=None) event_type='interaction.created' event_id=None metadata=None
interaction_id='ChBlNzBmN2NmODJjMTZhNWRjEAgaATAqBG1haW4' status='in_progress' event_id=None event_type='interaction.status_update' metadata=None
index=0 step=FunctionCallStep(arguments={}, id='call_130741', name='grep-search:searchGitHub', type='function_call') event_id=None event_type='step.start' metadata=None
delta=ArgumentsDelta(arguments='{"toolSummary":"GitHub search","explanation":"Searched for fibonacci pattern on GitHub","toolAction":"Searching GitHub","query":"def fibonacci("}', type='arguments_delta') index=0 event_id='v1_MF9mdW5jdGlvbkNhbGxfMTc5MDcyNDYzNzA3NTA2NA==' event_type='step.delta' metadata=None
index=0 event_id=None event_type='step.stop'

---
## Cleanup

Delete the test agents we created. Agent configurations persist until explicitly deleted.


### 5. Clean Up Custom Agents

To release resources and keep your Google Cloud project clean, delete the custom agent configurations when they are no longer needed. Deleted agents are removed permanently from the control plane.

In [ ]:
for agent_id_to_delete in [AGENT_ID, MCP_AGENT_ID]:
    try:
        response = client.agents.delete(id=agent_id_to_delete)
        print(f"Deleted agent: {agent_id_to_delete}")
    except Exception as e:
        print(f"Failed to delete {agent_id_to_delete}: {e}")

# Verify cleanup
response = client.agents.list()
remaining = [a.id for a in response.agents] if response.agents else []
print(f"\nRemaining agents: {remaining if remaining else 'None'}")


---
## Quick Reference

### Agents API

| Method | Code | Description |
|--------|------|-------------|
| Create | `client.agents.create(id=..., base_agent=..., ...)` | Create a reusable custom agent |
| List | `client.agents.list()` | List all agents (supports pagination) |
| Get | `client.agents.get(id="agent-id")` | Retrieve a specific agent by ID |
| Delete | `client.agents.delete(id="agent-id")` | Delete an agent |
| Update | *Not yet available in Python SDK* | Use REST API |

### Interactions API

| Method | Code | Description |
|--------|------|-------------|
| Create (streaming) | `client.interactions.create(agent=..., input=..., stream=True)` | Stream interaction events |
| Create (blocking) | `client.interactions.create(agent=..., input=...)` | Get final result |
| Session reuse | `environment="env_id_string"` | Reuse sandbox state |
| MCP override | `tools=[{"type": "mcp_server", ...}]` | Dynamic tool override |

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `agent` | Base agent name or custom agent ID |
| `input` | User prompt (string or structured content) |
| `environment` | `"remote"`, env_id string, or config dict |
| `stream` | `True` for SSE streaming, `False` for blocking |
| `background` | `True` to run in background |
| `store` | `True` to persist for later retrieval |
| `tools` | List of tool configs (MCP override, etc.) |
